##### Hyperparameter Tuning


Import required packages and libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from scipy.stats import uniform, randint

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

import warnings

Define the evaluate_model function as done in Model training

In [4]:
def evaluate_model(true, predicted, model, X_test):
    precision = precision_score(true, predicted , zero_division = 0)
    recall = recall_score(true, predicted , zero_division = 0)
    f1 = f1_score(true , predicted, zero_division = 0)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(true, y_pred_proba)
    pr_auc = average_precision_score(true, y_pred_proba)
    return precision, recall, f1, roc_auc, pr_auc

Define the classification function as defined in Model training

In [ ]:
def classification_hyperparam_tuning(X_train, y_train, X_test, y_test):

    y_train_blunder = y_train['is_blunder']
    y_test_blunder  = y_test['is_blunder']

    ratio = (y_train_blunder == 0).sum() / (y_train_blunder == 1).sum()
    
    # Models with hyperparameter tuning

    tuned_models = {

        "XGBoost": {
            "model": XGBClassifier(
                random_state=42,
                eval_metric='aucpr',
                verbosity=0,
            ),
            "params": {
                'n_estimators':     [200, 300, 500],
                'max_depth':        [3, 4, 5, 6],
                'learning_rate':    [0.01, 0.05, 0.1],
                'subsample':        [0.6, 0.7, 0.8, 1.0],
                'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
                'gamma':            [0, 0.1, 0.3, 0.5],
                'min_child_weight': [1, 3, 5, 10],
                'scale_pos_weight': [ratio * 0.5, ratio, ratio * 1.5, ratio * 2],
            }
        },

        "CatBoost": {
            "model": CatBoostClassifier(
                random_seed=42,
                verbose=False,
                thread_count=1,
            ),
            "params": {
                'iterations':       [200, 300, 500],
                'depth':            [4, 5, 6, 7, 8],
                'learning_rate':    [0.01, 0.05, 0.1, 0.2],
                'l2_leaf_reg':      [1, 3, 5, 7, 10],
                'border_count':     [32, 64, 128],
                'scale_pos_weight': [ratio * 0.5, ratio, ratio * 1.5, ratio * 2],
            }
        },

        "AdaBoost": {
            "model": AdaBoostClassifier(
                estimator=DecisionTreeClassifier(
                    class_weight='balanced',
                    max_depth=2,
                ),
                random_state=42,
                algorithm='SAMME',
            ),
            "params": {
                'n_estimators':            [100, 200, 300, 500],
                'learning_rate':           [0.01, 0.05, 0.1, 0.5, 1.0],
                'estimator__max_depth':    [1, 2, 3],
                'estimator__class_weight': ['balanced'],
            }
        },
    }
    results = []

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print("\nTraining models with hyperparameter tuning")
    for model_name, model_info in tuned_models.items():
        print(f"\n  {model_name}")

        random_search = RandomizedSearchCV(
            estimator=model_info["model"],
            param_distributions=model_info["params"],
            n_iter=75,                      
            cv=cv,                         
            scoring='average_precision',    # PR AUC
            refit=True,
            random_state=42,
            n_jobs=-1,
            verbose=1,
        )
        
        random_search.fit(X_train, y_train)
        best_model = random_search.best_estimator_
        
        y_train_pred = best_model.predict(X_train)
        y_test_pred = best_model.predict(X_test)
        
        accuracy_train = accuracy_score(y_train, y_train_pred)
        accuracy_test = accuracy_score(y_test, y_test_pred)
        
        train_precision, train_recall, train_f1, train_roc_auc, train_pr_auc = evaluate_model(
            y_train, y_train_pred, best_model, X_train
        )
        test_precision, test_recall, test_f1, test_roc_auc, test_pr_auc = evaluate_model(
            y_test, y_test_pred, best_model, X_test
        )
        
        results.append({
            'Model': model_name,
            'Train Accuracy': accuracy_train,
            'Test Accuracy': accuracy_test,
            'Test Precision': test_precision,
            'Test Recall': test_recall,
            'Test F1': test_f1,
            'Test ROC AUC': test_roc_auc,
            'Test PR AUC': test_pr_auc,
            'Tuned': 'Yes',
            'Best Params': str(random_search.best_params_)
        })
    
    # Create DataFrame
    df_results = pd.DataFrame(results)
    
    # Display main results table
    print("\n" + "=" * 150)
    print("MODEL COMPARISON - ALL METRICS")
    print("=" * 150)
    display_df = df_results.drop('Best Params', axis=1)
    print(display_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    
    # Display best parameters for tuned models
    print("\n" + "=" * 150)
    print("BEST HYPERPARAMETERS FOR TUNED MODELS")
    print("=" * 150)
    tuned_df = df_results[df_results['Tuned'] == 'Yes'][['Model', 'Best Params']]
    for idx, row in tuned_df.iterrows():
        print(f"\n{row['Model']}:")
        params = eval(row['Best Params'])
        for param, value in params.items():
            print(f"  - {param}: {value}")
    
    return df_results

In [ ]:
X_train=pd.read_csv('model_data/final_scaled_blunder_blitz_X_train.csv')
y_train=pd.read_csv('model_data/final_blunder_blitz_y_train.csv')

In [ ]:
X_test=pd.read_csv('model_data/final_scaled_blunder_blitz_X_test.csv')
y_test=pd.read_csv('model_data/final_blunder_blitz_y_test.csv')

Run the hyperparameter tuning

In [8]:
classification_hyperparam_tuning(X_train, y_train, X_test, y_test)


Training models with hyperparameter tuning

  XGBoost
Fitting 5 folds for each of 75 candidates, totalling 375 fits

  CatBoost
Fitting 5 folds for each of 75 candidates, totalling 375 fits

  AdaBoost
Fitting 5 folds for each of 60 candidates, totalling 300 fits


c:\Users\steve\anaconda3\envs\chess\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 60 is smaller than n_iter=75. Running 60 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\steve\anaconda3\envs\chess\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



MODEL COMPARISON - ALL METRICS
   Model  Train Accuracy  Test Accuracy  Test Precision  Test Recall  Test F1  Test ROC AUC  Test PR AUC Tuned
 XGBoost          0.9385         0.9288          0.2942       0.8970   0.4431        0.9747       0.5381   Yes
CatBoost          0.9069         0.9055          0.2450       0.9576   0.3901        0.9785       0.5897   Yes
AdaBoost          0.8787         0.8804          0.2074       0.9879   0.3428        0.9601       0.3374   Yes

BEST HYPERPARAMETERS FOR TUNED MODELS

XGBoost:
  - subsample: 0.7
  - scale_pos_weight: 15.307866868381241
  - n_estimators: 300
  - min_child_weight: 3
  - max_depth: 4
  - learning_rate: 0.05
  - gamma: 0
  - colsample_bytree: 0.8

CatBoost:
  - scale_pos_weight: 30.615733736762483
  - learning_rate: 0.05
  - l2_leaf_reg: 5
  - iterations: 200
  - depth: 5
  - border_count: 128

AdaBoost:
  - n_estimators: 100
  - learning_rate: 0.05
  - estimator__max_depth: 3
  - estimator__class_weight: balanced

TOP 5 MODELS BY

,Model,Train Accuracy,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC AUC,Test PR AUC,Tuned,Best Params
0,XGBoost,0.938463,0.928804,0.294235,0.896970,0.443114,0.974699,0.538055,Yes,"{'subsample': 0.7, 'scale_pos_weight': np.floa..."
1,CatBoost,0.906881,0.905455,0.244961,0.957576,0.390123,0.978458,0.589706,Yes,{'scale_pos_weight': np.float64(30.61573373676...
2,AdaBoost,0.878744,0.880383,0.207379,0.987879,0.342797,0.960116,0.337397,Yes,"{'n_estimators': 100, 'learning_rate': 0.05, '..."


Best model is XGBoost

In [ ]:
# ======================================================================================================================================================
# MODEL COMPARISON - ALL METRICS
# ======================================================================================================================================================
#    Model  Train Accuracy  Test Accuracy  Test Precision  Test Recall  Test F1  Test ROC AUC  Test PR AUC Tuned
#  XGBoost          0.9385         0.9288          0.2942       0.8970   0.4431        0.9747       0.5381   Yes
# CatBoost          0.9069         0.9055          0.2450       0.9576   0.3901        0.9785       0.5897   Yes
# AdaBoost          0.8787         0.8804          0.2074       0.9879   0.3428        0.9601       0.3374   Yes

# ======================================================================================================================================================
# BEST HYPERPARAMETERS FOR TUNED MODELS
# ======================================================================================================================================================

# XGBoost:
#   - subsample: 0.7
#   - scale_pos_weight: 15.307866868381241
#   - n_estimators: 300
#   - min_child_weight: 3
#   - max_depth: 4
#   - learning_rate: 0.05
#   - gamma: 0
#   - colsample_bytree: 0.8